In [1]:
import torch
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from torchvision import datasets
from torch.utils.data import Dataset
import numpy as np
import pandas as pd
from modules.base_utils.model.resnet import resnet32
from torchvision import transforms

from modules.base_utils.datasets import get_matching_datasets, pick_poisoner
from modules.base_utils.datasets import (
    load_dataset, PoisonedDataset, TRANSFORM_TEST_XY)
from modules.base_utils.util import clf_eval

import copy

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

# Loading the dataset

In [2]:
raw_train = datasets.CIFAR10("./data/data_cifar10", train=True, download=False)  # PIL images
classes = raw_train.classes        # ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

stripe_poisoner = pick_poisoner("1xs", "cifar", target_label=4)       # sinusoidal trigger, truck -> deer
poison_train, clean_train, test_set, poison_test, _ = get_matching_datasets("cifar", stripe_poisoner, 9)

train_loader = DataLoader(clean_train, batch_size=256, shuffle=True,  num_workers=4)
test_loader  = DataLoader(test_set,    batch_size=1000, shuffle=False, num_workers=4)

Files already downloaded and verified
Files already downloaded and verified


# Identifying source-class examples

In [3]:
classes = raw_train.classes
source_class = classes.index("truck")   # 9
target_class = classes.index("deer")    # 4

all_targets = torch.tensor([clean_train[i][1] for i in range(len(clean_train))])

# indices of images in the clean training set that belong to the source class (truck)
source_idx = (all_targets == source_class).nonzero().squeeze(1)

n_source = len(source_idx)

# Soft labels for the source class

In [16]:
num_classes = len(classes)

init_logits = torch.zeros(len(clean_train), num_classes)
for i in range(len(clean_train)):
    init_logits[i, clean_train[i][1]] = 10.0

label_logits = nn.Parameter(init_logits.to(device))
label_optimizer = torch.optim.Adam([label_logits], lr=100)

tau = 1.0

# Loss function

In [17]:
def soft_ce(logits, soft_targets):
    log_probs = F.log_softmax(logits, dim=1)
    return -(soft_targets * log_probs).sum(dim=1).mean()

The labels are chosen such that PTA + $\lambda$ CTA is the largest.

Assuming that $f_{\theta}$ is the neural network, $\text{PTA} = \frac{1}{n} \sum_{(x, y), y = y_{\text{source}}} 1_{f_{\theta}(T(x)) = y_{target}}$. Here, $n$ is the number of images belonging to $y_{\text{source}}$

To make this function differentiable, we need to make it smoother. So we use cross entropy loss where $y_{\text{source}}$ and $y_{\text{target}}$ are swapped.

$\text{PTA} = -\frac{1}{n} \sum_{(x, y), y = y_{\text{source}}} (y_{\text{target}} \log{p_{p_\text{source}}} + y_{\text{source}} \log{p_{p_\text{target}}} + \sum_{i \neq \text{source}, i \neq \text{target}} y_i \log(pi))$

In this expression $p_i$ is the probability the model $f_{\theta}$ assigns to the class $i$.

In [18]:
from torch.utils.data import Subset

poison_subset = Subset(poison_train, range(len(clean_train), len(poison_train)))

In [19]:
class IndexedDataset(Dataset):
    def __init__(self, base):
        self.base = base
    def __len__(self):
        return len(self.base)
    def __getitem__(self, i):
        img, hard_label = self.base[i]
        # if the image is poisoned, then return the index of the corresponding clean image instead
        return img, hard_label, i

indexed_train = IndexedDataset(clean_train)
train_loader = DataLoader(indexed_train, batch_size=256, shuffle=True, num_workers=4)

indexed_poisoned_trained = IndexedDataset(poison_subset)  # only the poisoned images
poisoned_train_loader = DataLoader(indexed_poisoned_trained, batch_size=256, shuffle=True, num_workers=4)

The DataLoader shuffles the data, that's why it is necessary to return the index as well, so we can retrieve the soft labels corresponding to the image that we're seeing.

Idea: for n rounds alternate training a ConvNet with the current labels and updating the labels
- the conv net trains on the current soft labels for $E$ epochs
- the conv net parameters are considered fixed, and the labels are updated using the cross entropy loss function

At each round, the model is reinitialized.

In [20]:
def train_one_epoch(model, loader, optimizer, label_logits, idx_to_row, tau, device):
    model.train()
    total_loss, correct, n = 0, 0, 0
    for imgs, hard_labels, idx in loader:
        imgs, hard_labels, idx = imgs.to(device), hard_labels.to(device), idx.to(device)

        targets = label_logits[idx].detach()
        with torch.no_grad():
            targets = F.softmax(targets, dim=1)
        logits = model(imgs)
        loss = soft_ce(logits, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(hard_labels)
        correct += (logits.argmax(dim=1) == targets.argmax(dim=1)).sum().item()
        n += len(hard_labels)

    print(f"train loss: {total_loss/n:.4f}, train_acc: {correct/n:.4f}")

In [31]:
def pta_target_labels(n, source_class, target_class, num_classes, device):
    # the "swapped" one-hot used inside the CE formula you wrote
    t = torch.zeros(n, num_classes, device=device)
    t[:, target_class] = 1.0   # reward the model predicting target_class
    return t

def outer_step(model, label_logits, clean_data_loader, poisoned_data_loader, source_class, target_class,
               num_classes, tau, lam, label_optimizer):
    model.eval()
    for p in model.parameters():
        p.requires_grad_(False)

    label_optimizer.zero_grad()

    total_pta, total_cta = 0.0, 0.0

    """

    for imgs, hard_labels, idx in clean_data_loader:
        imgs, hard_labels, idx = imgs.to(device), hard_labels.to(device), idx.to(device)

        targets = label_logits[idx]
        logits = model(imgs)
        soft_targets = F.softmax(targets, dim=1)
        loss = soft_ce(logits, soft_targets)

        # compute CTA
        with torch.no_grad():
            cta = (logits.argmax(dim=1) == hard_labels).float().mean().item()

        total_cta += cta * len(hard_labels)

        # compute gradient of loss w.r.t. label_logits
        loss.backward()

    """

    for imgs, hard_labels, idx in poisoned_data_loader:
        idx = source_idx[idx]  # map poisoned indices to clean indices
        imgs, hard_labels, idx = imgs.to(device), hard_labels.to(device), idx.to(device)

        soft_targets = F.softmax(label_logits[idx], dim=1)

        swapped = soft_targets.clone()
        swapped[:, [source_class, target_class]] = soft_targets[:, [target_class, source_class]]

        targets = label_logits[idx]
        logits = model(imgs)
        
        loss = 100 * soft_ce(logits, swapped)

        # compute PTA
        with torch.no_grad():
            pta = (logits.argmax(dim=1) == target_class).float().mean().item()
        total_pta += pta * len(hard_labels)

        # compute gradient of loss w.r.t. label_logits
        loss.backward()

    label_optimizer.step()

    for p in model.parameters():
        p.requires_grad_(True)

    return total_pta / 5000, total_cta / 50000

In [32]:
def label_stats(label_logits, orig_hard, source_idx, source_class, target_class, prev_probs=None):
    with torch.no_grad():
        probs = F.softmax(label_logits, dim=1)
        hard = probs.argmax(1)
        hard = hard.to(device)
        orig_hard = orig_hard.to(device)
        n = len(hard)
        ent = -(probs * probs.clamp_min(1e-12).log()).sum(1)

        s = {
            "p_true_mean":   probs[torch.arange(n), orig_hard].mean().item(),
            "entropy_mean":  ent.mean().item(),
            "n_flipped":     (hard != orig_hard).sum().item(),      # argmax differs from original label
            "src_p_source":  probs[source_idx, source_class].mean().item(),
            "src_p_target":  probs[source_idx, target_class].mean().item(),
            "logit_absmax":  label_logits.abs().max().item(),
        }
        if label_logits.grad is not None:
            g = label_logits.grad
            s["grad_norm"] = g.norm().item()
            s["rows_with_grad"] = (g.abs().sum(1) > 0).sum().item()
        if prev_probs is not None:
            s["mean_abs_change"] = (probs - prev_probs).abs().sum(1).mean().item()  # L1 per row
        # where do flipped labels go?
        flipped = hard != orig_hard
        s["flip_dest_hist"] = torch.bincount(hard[flipped], minlength=10).tolist()
        return s, probs

In [33]:
model = resnet32(10).to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)

R = 20          # outer rounds
E = 5         # inner epochs per round
K = 1         
lam = 1.0

for r in range(R):
    print(f"=== round {r} ===")
    model = resnet32(10).to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
    for _ in range(E):
        train_one_epoch(model, train_loader, optimizer, label_logits, _, tau, device)

    pta_hist, cta_hist = [], []
    for _ in range(K):
        pta_v, cta_v = outer_step(model, label_logits, train_loader, poisoned_train_loader, source_class, target_class, num_classes, tau, lam, label_optimizer)
        pta_hist.append(pta_v); cta_hist.append(cta_v)

    s, probs = label_stats(label_logits, all_targets, source_idx, source_class, target_class)
    print(f"label stats: {s}")
    print(f"round {r}: PTA={sum(pta_hist)/K:.4f} CTA={sum(cta_hist)/K:.4f}")


=== round 0 ===


train loss: 1.4269, train_acc: 0.4703
train loss: 0.9693, train_acc: 0.6481
train loss: 0.7729, train_acc: 0.7207
train loss: 0.6743, train_acc: 0.7565
train loss: 0.6207, train_acc: 0.7759
label stats: {'p_true_mean': 0.8523199558258057, 'entropy_mean': 1.2900509993585274e-08, 'n_flipped': 7384, 'src_p_source': 0.05379999801516533, 'src_p_target': 0.0026000000070780516, 'logit_absmax': 1045.1990966796875, 'grad_norm': 1.4153303709463216e-05, 'rows_with_grad': 1162, 'flip_dest_hist': [301, 1953, 421, 1018, 173, 239, 876, 148, 2248, 7]}
round 0: PTA=0.0462 CTA=0.0000
=== round 1 ===
train loss: 1.4454, train_acc: 0.4663
train loss: 1.0222, train_acc: 0.6268
train loss: 0.8085, train_acc: 0.7054
train loss: 0.6977, train_acc: 0.7474
train loss: 0.6252, train_acc: 0.7748
label stats: {'p_true_mean': 0.8523199558258057, 'entropy_mean': 1.3231813866809716e-08, 'n_flipped': 7384, 'src_p_source': 0.05379999801516533, 'src_p_target': 0.0026000000070780516, 'logit_absmax': 1061.5142822265625, '

KeyboardInterrupt: 